# SARIMA Parameter Tuning

Grid-search over `(p,d,q) × (P,D,Q,24)` combinations using a 48-hour holdout backtest on the EIA load data.

**How to use:**
1. Run **Cell 1** to install dependencies.
2. Run **Cell 2** to upload your `load_hourly.parquet` file.
3. Edit the `CONFIGURATION` cell to choose regions and the parameter search space.
4. Run all remaining cells. Results are sorted by MAPE ascending (lower is better).

In [ ]:
# Cell 1 — Install dependencies
!pip install -q statsmodels pandas pyarrow tqdm matplotlib

In [ ]:
# Cell 2 — Upload the parquet file
from google.colab import files
import io

print("Upload your load_hourly.parquet file:")
uploaded = files.upload()

import pandas as pd

filename = list(uploaded.keys())[0]
load_df = pd.read_parquet(io.BytesIO(uploaded[filename]))
load_df["period"] = pd.to_datetime(load_df["period"], utc=True)
load_df["value"] = pd.to_numeric(load_df["value"], errors="coerce")

print(f"\nLoaded {len(load_df):,} rows")
print(f"Date range : {load_df['period'].min()} → {load_df['period'].max()}")
print(f"Regions    : {sorted(load_df['region'].unique())}")
load_df.head()

In [ ]:
# ──────────────────────────────────────────────
# CONFIGURATION  ← edit this cell
# ──────────────────────────────────────────────

# Regions to include in the search. Set to None to use all regions.
REGIONS = None  # e.g. ["CISO", "ERCO"]

# How many recent hours to hold out for evaluation
HOLDOUT_HOURS = 168

# Max training history (None = use everything available)
MAX_TRAIN_HOURS = 1440  # ~60 days

# Minimum non-null observations required to fit the model
MIN_OBSERVATIONS = 96

# Parameter search space
P_VALUES = [0, 1, 2]   # AR order
D_VALUES = [1]          # differencing (1 is almost always right for load)
Q_VALUES = [0, 1, 2]   # MA order
SP_VALUES = [0, 1]     # seasonal AR order
SD_VALUES = [1]         # seasonal differencing
SQ_VALUES = [0, 1]     # seasonal MA order
SEASONAL_PERIOD = 24   # 24 = daily cycle in hourly data
# ──────────────────────────────────────────────

In [ ]:
# Core SARIMA fitting + forecast helper (mirrors src/models/sarima.py)
import warnings
import numpy as np
from dataclasses import dataclass
from statsmodels.tsa.statespace.sarimax import SARIMAX


@dataclass
class SarimaConfig:
    order: tuple = (1, 1, 1)
    seasonal_order: tuple = (1, 1, 1, 24)
    min_observations: int = MIN_OBSERVATIONS


def fit_and_forecast(series: pd.Series, horizon: int, config: SarimaConfig):
    """Return (predictions, info_dict). Falls back to persistence on short/failed series."""
    clean = series.sort_index().astype(float).asfreq("h")
    if clean.isna().all():
        return [float("nan")] * horizon, {}
    clean = clean.interpolate(limit_direction="both").ffill().bfill()
    if len(clean) < config.min_observations:
        last = float(clean.iloc[-1])
        return [last] * horizon, {}
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = SARIMAX(
                clean,
                order=config.order,
                seasonal_order=config.seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            fit = model.fit(disp=False)
        preds = list(fit.forecast(steps=horizon).astype(float))
        info = {"aic": fit.aic, "bic": fit.bic, "hqic": fit.hqic, "llf": fit.llf}
        return preds, info
    except Exception as e:
        return [float("nan")] * horizon, {"error": str(e)}


print("Helper functions defined.")

In [ ]:
# Metric helpers

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, float), np.array(y_pred, float)
    mask = y_true != 0
    if not mask.any():
        return float("nan")
    return float(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]).mean() * 100)


def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, float), np.array(y_pred, float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    if not mask.any():
        return float("nan")
    return float((np.abs(y_true[mask] - y_pred[mask]) / denom[mask]).mean() * 100)


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.array(y_true, float) - np.array(y_pred, float)) ** 2)))


def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.array(y_true, float) - np.array(y_pred, float))))


print("Metric helpers defined.")

In [ ]:
# Grid search
import itertools
from tqdm.auto import tqdm

group_cols = ["region", "series"]

# Filter regions
df = load_df.copy()
if REGIONS is not None:
    df = df[df["region"].isin(REGIONS)].copy()

# Build all parameter combos
param_grid = list(itertools.product(P_VALUES, D_VALUES, Q_VALUES, SP_VALUES, SD_VALUES, SQ_VALUES))
print(f"{len(param_grid)} parameter combinations × {df.groupby(group_cols).ngroups} series")
print(f"Total fits: {len(param_grid) * df.groupby(group_cols).ngroups}\n")

results = []

for p, d, q, sp, sd, sq in tqdm(param_grid, desc="Parameter combos"):
    config = SarimaConfig(
        order=(p, d, q),
        seasonal_order=(sp, sd, sq, SEASONAL_PERIOD),
    )
    label = f"({p},{d},{q})×({sp},{sd},{sq},{SEASONAL_PERIOD})"

    series_results = []

    for group_keys, gdf in df.groupby(group_cols):
        region, series_name = group_keys

        ts = gdf.set_index("period")["value"]
        ts = ts[~ts.index.duplicated(keep="last")].sort_index().asfreq("h")
        non_null = ts.dropna()

        if len(non_null) <= HOLDOUT_HOURS:
            continue

        # Build train / test split
        test_start = non_null.index.max() - pd.Timedelta(hours=HOLDOUT_HOURS - 1)
        train_end = test_start - pd.Timedelta(hours=1)
        train_ts = ts.loc[:train_end]
        if MAX_TRAIN_HOURS is not None:
            cutoff = train_end - pd.Timedelta(hours=MAX_TRAIN_HOURS - 1)
            train_ts = train_ts.loc[cutoff:train_end]

        y_true = non_null.loc[test_start:].astype(float).values[:HOLDOUT_HOURS]
        preds, info = fit_and_forecast(train_ts, HOLDOUT_HOURS, config)
        y_pred = np.array(preds, float)

        if np.isnan(y_pred).any() or len(y_true) != len(y_pred):
            continue

        series_results.append({
            "region": region,
            "series": series_name,
            "mae": mae(y_true, y_pred),
            "rmse": rmse(y_true, y_pred),
            "mape": mape(y_true, y_pred),
            "smape": smape(y_true, y_pred),
            "aic": info.get("aic"),
            "bic": info.get("bic"),
        })

    if not series_results:
        continue

    sdf = pd.DataFrame(series_results)
    results.append({
        "params": label,
        "p": p, "d": d, "q": q,
        "P": sp, "D": sd, "Q": sq,
        "n_series": len(sdf),
        "mae": sdf["mae"].mean(),
        "rmse": sdf["rmse"].mean(),
        "mape": sdf["mape"].mean(),
        "smape": sdf["smape"].mean(),
        "aic_mean": sdf["aic"].mean(),
        "bic_mean": sdf["bic"].mean(),
    })

results_df = pd.DataFrame(results).sort_values("mape").reset_index(drop=True)
print(f"\nDone. {len(results_df)} parameter combinations evaluated successfully.")

In [ ]:
# Results table — sorted by MAPE (lower is better)
display_cols = ["params", "n_series", "mae", "rmse", "mape", "smape", "aic_mean", "bic_mean"]

styled = (
    results_df[display_cols]
    .style
    .format({"mae": "{:.1f}", "rmse": "{:.1f}", "mape": "{:.3f}%", "smape": "{:.3f}%",
             "aic_mean": "{:.0f}", "bic_mean": "{:.0f}"})
    .background_gradient(subset=["mape"], cmap="RdYlGn_r")
    .set_caption(f"Grid search results — {HOLDOUT_HOURS}h holdout, sorted by MAPE ascending")
)
display(styled)

In [ ]:
# Best parameters summary
best = results_df.iloc[0]
print("=" * 60)
print("BEST PARAMETERS")
print("=" * 60)
print(f"  SARIMA order         : ({int(best.p)}, {int(best.d)}, {int(best.q)})")
print(f"  Seasonal order       : ({int(best.P)}, {int(best.D)}, {int(best.Q)}, {SEASONAL_PERIOD})")
print(f"  Avg MAPE across all series : {best.mape:.3f}%")
print(f"  Avg MAE  across all series : {best.mae:.1f}")
print(f"  Avg RMSE across all series : {best.rmse:.1f}")
print()
print("To apply these in production, set the following in your .env:")
print(f"  SARIMA_P={int(best.p)}")
print(f"  SARIMA_D={int(best.d)}")
print(f"  SARIMA_Q={int(best.q)}")
print(f"  SARIMA_SP={int(best.P)}")
print(f"  SARIMA_SD={int(best.D)}")
print(f"  SARIMA_SQ={int(best.Q)}")

In [ ]:
# Bar chart — top 15 configs by MAPE
import matplotlib.pyplot as plt

top15 = results_df.head(15).copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ["#2ecc71" if i == 0 else "#95a5a6" for i in range(len(top15))]

axes[0].barh(top15["params"][::-1], top15["mape"][::-1], color=colors[::-1])
axes[0].set_xlabel("MAPE (%) — lower is better")
axes[0].set_title("Top 15 configurations by MAPE")
axes[0].axvline(top15["mape"].iloc[0], color="green", linestyle="--", alpha=0.6)

axes[1].barh(top15["params"][::-1], top15["rmse"][::-1], color=colors[::-1])
axes[1].set_xlabel("RMSE — lower is better")
axes[1].set_title("Top 15 configurations by RMSE")

fig.suptitle(f"SARIMA Grid Search — {HOLDOUT_HOURS}h holdout", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("sarima_grid_search.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sarima_grid_search.png")

In [ ]:
# Forecast vs actual plots for the best config
# Shows up to 4 regions side by side

best_config = SarimaConfig(
    order=(int(best.p), int(best.d), int(best.q)),
    seasonal_order=(int(best.P), int(best.D), int(best.Q), SEASONAL_PERIOD),
)

plot_regions = sorted(df["region"].unique())[:4]
fig, axes = plt.subplots(len(plot_regions), 1, figsize=(14, 4 * len(plot_regions)), sharex=False)
if len(plot_regions) == 1:
    axes = [axes]

for ax, region in zip(axes, plot_regions):
    region_df = df[df["region"] == region]
    # Use the first series found for this region
    series_name = region_df["series"].iloc[0]
    gdf = region_df[region_df["series"] == series_name]

    ts = gdf.set_index("period")["value"]
    ts = ts[~ts.index.duplicated(keep="last")].sort_index().asfreq("h")
    non_null = ts.dropna()

    if len(non_null) <= HOLDOUT_HOURS:
        ax.set_title(f"{region} — insufficient data")
        continue

    test_start = non_null.index.max() - pd.Timedelta(hours=HOLDOUT_HOURS - 1)
    train_end = test_start - pd.Timedelta(hours=1)
    train_ts = ts.loc[:train_end]
    if MAX_TRAIN_HOURS is not None:
        cutoff = train_end - pd.Timedelta(hours=MAX_TRAIN_HOURS - 1)
        train_ts = train_ts.loc[cutoff:]

    y_true = non_null.loc[test_start:].values[:HOLDOUT_HOURS]
    preds, _ = fit_and_forecast(train_ts, HOLDOUT_HOURS, best_config)
    y_pred = np.array(preds, float)

    # Show 7 days of history + the holdout window
    history_start = train_end - pd.Timedelta(days=7)
    history_ts = ts.loc[history_start:train_end]
    holdout_idx = pd.date_range(start=test_start, periods=HOLDOUT_HOURS, freq="h")

    ax.plot(history_ts.index, history_ts.values, color="steelblue", label="actual (history)")
    ax.plot(holdout_idx, y_true, color="steelblue", linestyle="--", label="actual (holdout)")
    ax.plot(holdout_idx, y_pred, color="tomato", linewidth=2, label=f"forecast {best.params}")
    ax.axvline(test_start, color="gray", linestyle=":", alpha=0.8)
    ax.set_title(f"{region} — {series_name}  |  MAPE={mape(y_true, y_pred):.2f}%")
    ax.set_ylabel("MW")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Time (UTC)")
fig.suptitle(f"Best config: SARIMA{best.params} — {HOLDOUT_HOURS}h holdout", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("sarima_best_forecast.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sarima_best_forecast.png")

In [ ]:
# Save the full results table as a CSV for reference
results_df.to_csv("sarima_grid_search_results.csv", index=False)
print("Results saved to sarima_grid_search_results.csv")

# Download all outputs
from google.colab import files
files.download("sarima_grid_search_results.csv")
files.download("sarima_grid_search.png")
files.download("sarima_best_forecast.png")